# שיעור 33 — פרספטרון: החלטה, עדכון משקולות וקו הפרדה

**הקשר ללמידת מכונה:** דודו רוצה שמודל ישנה כלל החלטה אחרי טעות,ולא רק יחפש שכנים. נראה כיצד תחזית וסימן טעות משנים משקולות.

**עוגן:** [סיכום מפגש 11](../../13-Perceptron-SVM/Meeting-11-Didactic-Summary.md). כלל המצגת z>0 נשמר. דוגמאות AND/OR מקוריות שלנו מחליפות את נתוני עיר/יער; לא משנים את פתרון המקור,שבסיכום צוינו בו אי־התאמות.

המחברת היא התאמה דידקטית חדשה. אין קובץ מקור חדש מהמרצה ולכן אין Exact-Original מלאכותי. אין לייחס לה אישור כיסוי מחוון עדכני.

## למה לומדים את זה עכשיו?
האם קיוסק יציע מבצע כשיש מלאי עודף וגם נותר מעט זמן? נבדיל כלל נתון ממשקולות שנלמדות.
**בתהליך הפרויקט:** מודל ולמידה.
**מסגרת פעילה:** 2.5 שעות לימוד — 112.5 דקות, כולל דוגמה ותרגול עצמאי. המורה בוחר את משימות הליבה שבמדריך; שאר הפעילויות לחזרה או לעיון, ללא תוספת שעות.

## מושגים וידע קודם
z אינו הסתברות. eta הוא קצב למידה; epoch הוא מעבר על כל דוגמאות האימון. מעדכנים גם b,ורק אחרי חישוב הטעות. נשתמש במקסימום מעברים מוגדר,בלי while. בנתונים שאינם ניתנים להפרדה ליניארית אין הבטחת עצירה עם אפס טעויות. טבלת אמת אינה Dataset אמיתי ואין כאן הערכת הכללה.

## הסבר מתמטי
z=w₁x₁+w₂x₂+b; תחזית 1 אם z>0,אחרת 0. e=y−ŷ. עדכון w←w+ηex ו־b←b+ηe. למשל w=(0,0),b=0,x=(1,1),y=1 נותנים ŷ=0; עבור η=.5 נקבל w=(.5,.5),b=.5. קו ההפרדה z=0. פרספטרון יחיד מפריד ליניארית; XOR אינו ניתן להפרדה בקו יחיד.

## החלטה לפני אימון
חישוב סימן אינו תהליך לימוד. בקו z=0 כלל המקור מחזיר 0 ולא 1.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# טבלת אמת מומצאת: AND
X=np.array([[0.,0.],[0.,1.],[1.,0.],[1.,1.]])
y=np.array([0,0,0,1])

fixed_w=np.array([1.,1.]);fixed_b=-1.5
print("z for (1,1):",np.dot(fixed_w,[1,1])+fixed_b)
print("Boundary z=0 predicts:",1 if 0>0 else 0)

## עדכון ואימון
מחשבים דוגמה אחת ביד לפני הרצת הפונקציה; אפשר להשתמש בתבנית נתונה ולמלא עמודות עקיבה. האינדקס epoch מתחיל באפס.

In [ ]:
def predict_perceptron(X,w,b):
    output=[]
    for row in X:
        z=float(np.dot(w,row)+b)
        output.append(1 if z>0 else 0)
    return np.array(output)

def train_perceptron(X,y,learning_rate=0.5,max_epochs=40):
    w=np.zeros(X.shape[1],dtype=float)
    b=0.0
    trace=[]
    for epoch in range(max_epochs):
        mistakes=0
        for i in range(len(X)):
            before=w.copy()
            b_before=b
            z=float(np.dot(w,X[i])+b)
            predicted=1 if z>0 else 0
            error=int(y[i])-predicted
            if error!=0:
                w=w+learning_rate*error*X[i]
                b=b+learning_rate*error
                mistakes=mistakes+1
            trace.append([epoch,i,z,predicted,int(y[i]),error,
                          before[0],before[1],b_before,w[0],w[1],b])
        if mistakes==0:
            break
    columns=["epoch","row","z","pred","actual","error","w1_before","w2_before","b_before","w1_after","w2_after","b_after"]
    return w,b,pd.DataFrame(trace,columns=columns)

w,b,trace=train_perceptron(X,y)
predictions=predict_perceptron(X,w,b)
display(trace.head(12))
print("Weights:",w,"Bias:",b,"Training predictions:",predictions)


## קו ומגבלה
מציירים z=0 כשהמשקל השני אינו אפס. אם הוא אפס ומשקל ראשון אינו אפס הקו אנכי; במשקולות אפס אין קו יחיד. הקוד מטפל במקרה הזה.

In [ ]:
plt.figure()
plt.scatter(X[:,0],X[:,1],c=y,cmap="coolwarm",vmin=0,vmax=1)
grid=np.array([-0.2,1.2])
if abs(w[1])>1e-12:
    plt.plot(grid,-(w[0]*grid+b)/w[1],color="black")
elif abs(w[0])>1e-12:
    plt.axvline(-b/w[0],color="black")
plt.xlim(-0.2,1.2);plt.ylim(-0.2,1.2)
plt.xlabel("x1");plt.ylabel("x2");plt.title("Synthetic training data — perceptron boundary")
plt.tight_layout();plt.show()
xor_y=np.array([0,1,1,0])
print("XOR labels:",xor_y)
# לא דורשים מהאלגוריתם למצוא קו שאינו קיים; ממשיכים לרשת בשיעור 35


## מסקנה ובדיקת הבנה
תעדו מה נכנס,מה חושב,מה התקבל ומה אי אפשר להסיק. כתבו שני משפטים על הקשר לפרויקט.
אין לערבב את כלל z>0 עם z≥0; לא מעדכנים לפני רישום התחזית; לא מדלגים על b. למתקשים מריצים מעבר אחד בלבד ביד ואז מקבלים פונקציה נתונה; למתקדמים בודקים תלות בסדר הדוגמאות. מושגי נוירון ביולוגי הם אנלוגיה,לא זהות.

**כרטיס יציאה:** הסבירו פעולה אחת ללא קריאת הקוד ומגבלה אחת של הדוגמה. דוגמאות מומצאות אינן ראיה על אוכלוסייה אמיתית.